# Quantum sine and cosine transforms

All four original examples use native FlagQuantum quantum functions. Retain RandomState seed 123 and sequential draws of 9, 7 and 8real values. Inputs are bound at synthesis time;no runtime binding,autograd or hardware claim is made.


In [ ]:
from math import pi, sqrt

import torch

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QBit,
    QNum,
    allocate,
    prepare_amplitudes,
    qct_qst_type1,
    qct_qst_type2,
    qct_type2,
    qfunc,
    qst_type2,
)


from examples.synthesis.functions.qct_qst.overview.main import dct, dst, original_data

cos_data, sin_data, data = original_data()
combined = torch.cat((cos_data, sin_data)) / sqrt(2)
print("Input lengths:", len(cos_data), len(sin_data), len(data))


## Native quantum functions

Each function prepares a real amplitude array then applies an existing high-level transform. The combinedType II example allocates and superposes its selector inside the same qfunc. No physical wire numbers or handwritten transform gates are needed.


In [ ]:
@qfunc
def type_one(data: list[float], x: Output[QNum]) -> None:
    prepare_amplitudes(data, 0, x)
    qct_qst_type1(x)


@qfunc
def type_two(data: list[float], x: Output[QNum], q: Output[QBit]) -> None:
    prepare_amplitudes(data, 0, x)
    allocate(q)
    H(q)
    qct_qst_type2(x, q)


@qfunc
def cosine(data: list[float], x: Output[QNum]) -> None:
    prepare_amplitudes(data, 0, x)
    qct_type2(x)


@qfunc
def sine(data: list[float], x: Output[QNum]) -> None:
    prepare_amplitudes(data, 0, x)
    qst_type2(x)


## Type I: cosine and+i times sine

The 9-elementDCT I and 7-elementDST I occupy separate blocks. The sine block retains relative phase+i. Source executable formulas use sine indices(j+1)(k+1);the introductory prose misses this shift.


In [ ]:
program_one = synth.synthesize(
    type_one,
    classical_inputs={"data": combined.tolist()},
    max_qubits=4,
    max_workspace=0,
    memory_limit_bytes=1024**2,
)
state_one = fq.run(program_one).statevector().flatten()
expected_one = torch.cat((dct(9, 1) @ cos_data, 1j * (dst(7, 1) @ sin_data))) / sqrt(2)
torch.testing.assert_close(state_one, expected_one, atol=1e-12, rtol=0)
print("Measured:", state_one)
print("Expected:", expected_one)


## CombinedType II

Use 3data bits plus selector q,with declared register order(x,q). Selector0 uses DCT II;selector 1 uses negativeDST II. The source prose dimension is inconsistent;the executable example fixes an 8-element transform per selector.


In [ ]:
program_two = synth.synthesize(
    type_two,
    classical_inputs={"data": data.tolist()},
    max_qubits=4,
    max_workspace=0,
    memory_limit_bytes=1024**2,
)
state_two = fq.run(program_two).statevector().flatten()
expected_two = (
    torch.stack((dct(8, 2) @ data, -(dst(8, 2) @ data)), dim=1).flatten() / sqrt(2)
).to(torch.complex128)
torch.testing.assert_close(state_two, expected_two, atol=1e-12, rtol=0)
print("Measured:", state_two)
print("Expected:", expected_two)


## Standalone cosineType II


In [ ]:
program_cosine = synth.synthesize(
    cosine,
    classical_inputs={"data": data.tolist()},
    max_qubits=3,
    max_workspace=0,
    memory_limit_bytes=1024**2,
)
state_cosine = fq.run(program_cosine).statevector().flatten()
expected_cosine = (dct(8, 2) @ data).to(torch.complex128)
torch.testing.assert_close(state_cosine, expected_cosine, atol=1e-12, rtol=0)
print("Measured:", state_cosine)
print("Expected:", expected_cosine)


## Standalone sineType II

This block has a positive sign. Native checks retain absolute complex phase within 1e-12;the source used separate block-phase alignment and 0.01tolerance. Readout here uses the existing statevector accessor;no bitstring decoding is needed.


In [ ]:
program_sine = synth.synthesize(
    sine,
    classical_inputs={"data": data.tolist()},
    max_qubits=3,
    max_workspace=0,
    memory_limit_bytes=1024**2,
)
state_sine = fq.run(program_sine).statevector().flatten()
expected_sine = (dst(8, 2) @ data).to(torch.complex128)
torch.testing.assert_close(state_sine, expected_sine, atol=1e-12, rtol=0)
print("Measured:", state_sine)
print("Expected:", expected_sine)
